In [16]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CocoDetection
from transformers import DetrForObjectDetection, DetrImageProcessor
from PIL import Image, ImageDraw

In [21]:
# Custom dataset wrapper for HuggingFace format
class CocoDetectionHFDataset(Dataset):
    def __init__(self, root, annFile, processor):
        self.coco_dataset = CocoDetection(root, annFile)
        self.processor = processor

    def __len__(self):
        return len(self.coco_dataset)

    def __getitem__(self, idx):
        image, target = self.coco_dataset[idx]
        annotations = [
            {
                "bbox": anno["bbox"],
                "category_id": anno["category_id"],
                "area": anno.get("area", anno["bbox"][2] * anno["bbox"][3]),  # width * height
                "iscrowd": anno.get("iscrowd", 0)
            } 
            for anno in target
        ]
        return image, {"image_id": idx, "annotations": annotations}


# Load image processor
image_processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')

# Dataset initialization
train_dataset = CocoDetectionHFDataset('C:/Users/mza0288/Datset/train2017', 'C:/Users/mza0288/Datset/annotations/instances_train2017.json', image_processor)
val_dataset = CocoDetectionHFDataset('C:/Users/mza0288/Datset/val2017', 'C:/Users/mza0288/Datset/annotations/instances_val2017.json', image_processor)

# Corrected collate function
def collate_fn(batch):
    images, targets = zip(*batch)
    encoding = image_processor(images=list(images), annotations=list(targets), return_tensors="pt")
    return encoding

# DataLoader setup
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [23]:
# Corrected collate function
def collate_fn(batch):
    images, targets = zip(*batch)
    encoding = image_processor(images=list(images), annotations=list(targets), return_tensors="pt")
    return encoding

# DataLoader setup
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

# Model definition
num_classes = 3  # two classes + background
model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

# Optimizer setup
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DetrForObjectDetection were not initialized from the model checkpoin

In [25]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model explicitly to GPU
model.to(device)

epochs = 10
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        # Ensure inputs are correctly moved to GPU
        pixel_values = batch['pixel_values'].to(device)
        labels = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in batch['labels']]
        
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")


Epoch [1/10], Loss: 4.3042
Epoch [2/10], Loss: 4.2759
Epoch [3/10], Loss: 3.7407
Epoch [4/10], Loss: 3.6151
Epoch [5/10], Loss: 3.6409
Epoch [6/10], Loss: 3.6746
Epoch [7/10], Loss: 3.7328
Epoch [8/10], Loss: 3.6005
Epoch [9/10], Loss: 3.5236
Epoch [10/10], Loss: 3.5059


In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'detr_instance_segmentation.pth')

In [30]:
# Prediction and Visualization
def predict(image_path, threshold=0.5):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    width, height = image.size
    results = image_processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=[(height, width)])[0]

    draw = ImageDraw.Draw(image)

    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i) for i in box.tolist()]
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], box[1]), f'{score:.2f}', fill='white')

    image.show()

# Example usage
predict('C:/Users/mza0288/Datset/val2017/tile_0_0_jpg.rf.2bc2a6231fb40041d7845e1918741a3d.jpg')